In [1]:
import os
import json
import pandas as pd
import numpy as np

In [2]:
import os
print(os.getcwd())

/Users/dilshanpathegamage/Projects/Personal Projects/SriLanka_Tourism_Prediction_ML_Project/scripts


In [3]:
BASE_PATH = "../raw_data"
OUTPUT_PATH = "../processed_data"

os.makedirs(OUTPUT_PATH, exist_ok=True)

YEARS = [2019, 2020, 2021, 2022, 2023, 2024]

In [4]:
def load_json_to_df(filepath):
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        return pd.DataFrame()
    with open(filepath, "r") as f:
        data = json.load(f)
    return pd.DataFrame(data["rows"], columns=data["columns"])

In [5]:
# --------------------------
# Tourist Arrivals By Month
# --------------------------
all_arrivals = []

for year in YEARS:
    path = f"{BASE_PATH}/{year}/Tourist_Arrivals_By_Month_data.json"
    df = load_json_to_df(path)
    df["Year"] = year
    all_arrivals.append(df)

arrivals_by_month = pd.concat(all_arrivals, ignore_index=True)

arrivals_by_month.to_csv(
    f"{OUTPUT_PATH}/Tourist_Arrivals_By_Month_2019_2024.csv",
    index=False
)

arrivals_by_month.head()

,Month,Number of tourist arrivals,Year
0,January,244239,2019
1,February,252033,2019
2,March,244328,2019
3,April,166975,2019
4,May,37802,2019


In [6]:

# --------------------------
# Tourist Arrivals By Country And Month
# --------------------------
all_country_month = []

for year in YEARS:
    path = f"{BASE_PATH}/{year}/Tourist_Arrivals_By_Country_And_Month_data.json"
    df = load_json_to_df(path)
    df["Year"] = year
    all_country_month.append(df)

country_month = pd.concat(all_country_month, ignore_index=True)

country_month.to_csv(
    f"{OUTPUT_PATH}/Tourist_Arrivals_By_Country_Month_2019_2024.csv",
    index=False
)

country_month.head()

,Country,January,February,March,April,May,June,July,August,September,October,November,December,Year
0,Afghanistan,49,47,59,38,2,12,47,54,50,22,55,38,2019
1,Albania,48,33,54,43,40,31,17,15,11,23,27,27,2019
2,Algeria,34,32,36,32,3,8,3,17,29,16,13,49,2019
3,Andorra,0,0,0,0,0,0,0,0,0,0,0,0,2019
4,Angola,0,0,0,0,0,0,0,0,0,0,0,0,2019


In [7]:
# --------------------------
# Top 10 Source Markets
# --------------------------
all_top10 = []

for year in YEARS:
    path = f"{BASE_PATH}/{year}/Top_10_Source_Markets_data.json"
    df = load_json_to_df(path)
    df["Year"] = year
    all_top10.append(df)

top10_markets = pd.concat(all_top10, ignore_index=True)

top10_markets.to_csv(
    f"{OUTPUT_PATH}/Top10_Source_Markets_2019_2024.csv",
    index=False
)

top10_markets.head()

,Country,Arrivals,Share,Year
0,India,355002,18.6,2019
1,United Kingdom,198776,10.4,2019
2,China,167863,8.8,2019
3,Germany,134899,7.0,2019
4,France,87623,4.6,2019


In [8]:
# --------------------------
# Accommodation By Province
# --------------------------

# Load all raw data
all_accommodation = []
for year in YEARS:
    path = f"{BASE_PATH}/{year}/Accommodations_By_Province_data.json"
    df = load_json_to_df(path)
    df["Year"] = year
    all_accommodation.append(df)

accommodation = pd.concat(all_accommodation, ignore_index=True)

# Fix 2019 inconsistent province names
mapping_2019 = {
    "Colombo City": "Western Province",
    "Greater Colombo": "Western Province",
    "South Coast": "Southern Province",
    "East Coast": "Eastern Province",
    "High Country": "Central Province",
    "Ancient Cities": "North Central Province",
    "North Region": "Northern Province",
    "All Regions": None
}

mask_2019 = accommodation["Year"] == 2019
accommodation.loc[mask_2019, "Province"] = accommodation.loc[mask_2019, "Province"].map(mapping_2019)
accommodation = accommodation[accommodation["Province"].notna()]

# Aggregate duplicates by summing Number of Rooms
accommodation = accommodation.groupby(["Province", "Year"], as_index=False).agg({
    "Number of Rooms": "sum"
})

# Create full Province × Year grid
provinces = accommodation["Province"].drop_duplicates()
full_grid = pd.DataFrame([(p, y) for p in provinces for y in YEARS], columns=["Province", "Year"])

# Merge with actual data
accommodation_full = full_grid.merge(accommodation, on=["Province", "Year"], how="left")

# Fill missing 2019 values with 0
accommodation_full.loc[accommodation_full["Year"] == 2019, "Number of Rooms"] = \
    accommodation_full.loc[accommodation_full["Year"] == 2019, "Number of Rooms"].fillna(0)

# Interpolate internal gaps only
accommodation_full["Number of Rooms"] = (
    accommodation_full.groupby("Province")["Number of Rooms"]
    .transform(lambda x: x.interpolate(method="linear"))
)

accommodation_full = accommodation_full.sort_values(["Year", "Province"])

# Save cleaned CSV
accommodation_full.to_csv(f"{OUTPUT_PATH}/Accommodation_By_Province_2019_2024.csv", index=False)


In [9]:
# --------------------------
# Attraction Revenue & Visitors
# --------------------------
attraction_rows = []
for year in YEARS:
    path = f"{BASE_PATH}/{year}/Location_Vs_Revenue_Vs_Visitors_Count_data.json"
    if os.path.exists(path):
        df = load_json_to_df(path)
        df["Year"] = year
        attraction_rows.append(df)

attraction_raw = pd.concat(attraction_rows, ignore_index=True)

place_mapping = {
    # Botanical Gardens
    "Avissawella Botanical Garden": "Avissawella",
    "Gampaha Botanical Garden": "Gampaha",
    "Ganewattha Botanical Garden": "Ganewattha",
    "Hakgala Botanical Garden": "Hakgala",
    "Mirijjawila": "Mirijawila",
    "Mirijawila Botanical Garden": "Mirijawila",
    "Peradeniya Botanical Garden": "Peradeniya",

    # Conservation Forests
    "Udawattakele Conservation Forest/ Kandy": "Udawattakele Conservation Forest",
    "Kandeela Forest": "Kande Ela Forest",
    "Knuckles Conservation": "Knuckles Conservation Forest",
    "Ritigala forest Monastery": "Ritigala Forest Monastery",
    "Piduruthalagala (Nuwaraeliya)": "Piduruthalagala",
    "Dolukanda (Kurunegala)": "Dolukanda",
    "Galwila Eco Park (Puttlam)": "Galwila Eco Park",
    "Mandaramnuwara (Kolaathanaella) (Nuwaraeliva)": "Mandaramnuwara",
    "Badagamuwa Ecological Zone (Kuruneoalal": "Badagamuwa Ecological Zone",
    "Badulla (Elle Gala)": "Badulla",
    "Nuwaragala (Ampara)": "Nuwaragala",

    # Cultural Triangle
    "Ibbankatuwa Ancient Bural Ground": "Ibbankatuwa Ancient Burial Ground",
    "Polonnaruwa Gal Viharaya,Museum & Kingdom": "Polonnaruwa",
    "Ramba Viharaya": "Ramba",
    "Ritigala forest Monastery": "Ritigala Forest Monastery",
    "Sigiriya Museum and Sigiriya Rock": "Sigiriya",
    "Ampara lahugala": "Lahugala",
    "Katharagama Museum": "Kataragama Museum",

    # Wildlife Parks
    "Eth Athurusevana": "Eth Athuru Sevana",
    "Eth Athurusevena": "Eth Athuru Sevana",
    "Hikkaduawa": "Hikkaduwa",
    "Udawalawa National Park": "Udawalawa",
    "Wilpattu National Park": "Wilpattu",
    "Yala National Park": "Yala",

    # Zoological Gardens
    "Dehiwala Zoo": "Dehiwala",
    "Pinnawala Elephant Orphanage": "Pinnawala",
    "Pinnawala Zoo": "Pinnawala",
    "Safari Park": "Ridiyagama Safari Park"
}

attraction_raw["Place"] = attraction_raw["Place"].replace(place_mapping)

# Get unique places & types
places = attraction_raw[["Type", "Place"]].drop_duplicates()

# Create full Place × Year grid
full_grid = places.assign(key=1).merge(pd.DataFrame({"Year": YEARS, "key": 1}), on="key").drop("key", axis=1)

# Merge with raw data
attraction_full = full_grid.merge(attraction_raw, on=["Type", "Place", "Year"], how="left")

# Sort and interpolate numeric values per Place
numeric_cols = ["Foreign Number", "Foreign Income", "Local Number", "Local Income"]
attraction_full = attraction_full.sort_values(["Place", "Year"])
attraction_full[numeric_cols] = attraction_full.groupby("Place")[numeric_cols].transform(
    lambda x: x.interpolate(method="linear").fillna(0)
)

# Final sort by Year, Type, Place
attraction_full = attraction_full.sort_values(["Year", "Type", "Place"])

# Ensure column order is exactly as required
columns_order = ["Type", "Place", "Year", "Foreign Number", "Foreign Income", "Local Number", "Local Income"]
attraction_full = attraction_full[columns_order]

# Save full CSV
attraction_full.to_csv(f"{OUTPUT_PATH}/Attraction_Revenue_Visitors_2019_2024.csv", index=False)

In [10]:
os.listdir(OUTPUT_PATH)

['Tourist_Arrivals_By_Month_2019_2024.csv',
 'Accommodation_By_Province_2019_2024.csv',
 'Top10_Source_Markets_2019_2024.csv',
 'Tourist_Arrivals_By_Country_Month_2019_2024.csv',
 'Attraction_Revenue_Visitors_2019_2024.csv',
 '.ipynb_checkpoints']